<a href="https://colab.research.google.com/github/prithwis/Centaur/blob/main/MOS_MonteCarlo_v0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![alt text](https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/images/CentaurBanner2.png)<br>


<hr>

[Prithwis Mukerjee](http://www.linkedin.com/in/prithwis)<br>

# MOS :: Middle East Oil Security
### An LLM-Based Geopolitical What-If Simulator

> **Action → Consequence → Reaction → Consequence → ...**

**MOS** explores how geopolitical situations may evolve when strategic actors respond to one another in an uncertain and changing world.

**Chanakya** proposes strategic alternatives.  
**Centaur** evaluates their relative likelihood.  
**Shakuni** -- the dice roller -- selects a path under uncertainty.  
**ZeitWorld** translates the selected Event into the next WorldState.

The cycle repeats, creating one possible **Scenario** from an initial hypothetical Trigger.

> **The objective is not to predict the future, but to explore plausible, path-dependent futures arising from strategic choices.**

In [1]:
!pip install -q python-docx
from IPython.display import Markdown, display
from datetime import datetime
from zoneinfo import ZoneInfo
import pytz
print('ॐ श्री सरस्वत्यै नमः',datetime.now(pytz.timezone('Asia/Calcutta')))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 13.1 MB/s eta 0:00:00
ॐ श्री सरस्वत्यै नमः 2026-10-09 12:38:34.888352+05:30


In [2]:
#!pip install -q python-docx
#from docx.oxml import OxmlElement
#from docx.oxml.ns import qn

In [3]:
!wget -O CentaurLogo.png -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/images/logo-S.png

!wget -O mos.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mos.py
!wget -O mosZeitWorld.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mosZeitWorld.py
!wget -O mosChanakya.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mosChanakya.py
!wget -O mosCentaur.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mosCentaur.py
!wget -O mosReporter.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mosReporter.py
!wget -O mosDOCx.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mosDOCx.py

from mos import *
from mosZeitWorld import *
from mosChanakya import *
from mosCentaur import *
from mosReporter import *
from mosDOCx import *

In [4]:
import importlib

import mos
import mosZeitWorld
import mosChanakya
import mosCentaur
import mosReporter
import mosDOCx

importlib.reload(mos)
importlib.reload(mosZeitWorld)
importlib.reload(mosChanakya)
importlib.reload(mosCentaur)
importlib.reload(mosReporter)
importlib.reload(mosDOCx)

from mos import *
from mosZeitWorld import *
from mosChanakya import *
from mosCentaur import *
from mosReporter import *

## Initialise Runtime Environment

Authenticate OpenAI, select the working LLM, mount Google Drive and define the paths to the MOS data, roles and prompts.

The directory listings provide a quick check that all required resources are available.

In [5]:
client = authenticateOpenAI()
model = "gpt-5.6-luna"

# Real-world evidence, LLM roles and prompts
# are stored persistently in Google Drive.

from google.colab import drive
drive.mount('/content/drive')

MOS_HOME = "/content/drive/MyDrive/Praxis/Course Geopolitics/Centaur Project/MOS/"
WF_PATH = f"{MOS_HOME}/WorldFacts"
ZW_PATH = f"{MOS_HOME}/ZeitWorld"
CH_PATH = f"{MOS_HOME}/Chanakya"
CT_PATH = f"{MOS_HOME}/Centaur"
PR_PATH = f"{MOS_HOME}/Reporter"
OP_PATH = f"{MOS_HOME}/Reports"

#!ls -l "{WORLDFACTS_DIR}"
#!ls -lh "{WORLDFACTS_DIR}/MD"
#!ls -lh "{WORLDFACTS_DIR}/NEWS"
#!ls -lh "{ZW_PATH}"
#!ls -lh "{CH_PATH}"
#!ls -lh "{CT_PATH}"

OpenAI authentication successful ✔
Logged in as Calcutta prithwis@yantrajaal.com
Mounted at /content/drive


In [6]:
!ls -l "{MOS_HOME}"

total 28
drwx------ 2 root root 4096 Oct  2 14:26 Centaur
drwx------ 2 root root 4096 Oct  2 00:08 Chanakya
drwx------ 2 root root 4096 Oct  3 09:51 Documentation
drwx------ 2 root root 4096 Oct  3 04:01 Reporter
drwx------ 2 root root 4096 Oct  8 00:07 Reports
drwx------ 2 root root 4096 Sep 29 01:26 WorldFacts
drwx------ 2 root root 4096 Sep 30 23:02 ZeitWorld


## Get World Facts

In [7]:
# Build WorldFactBase
#
# Static PDF-derived Markdown files and live Moneycontrol news
# are assembled into a single corpus and stored at location WF_PATH
# This data is used to initialise the WorldState W0
# No real-world information enters the simulation after W0.
# ------------------------------------------------------------

WorldFactBase = getWorldFactBase(WF_PATH)

Static sources : 10
Live sources   : 1
Characters     : 877,462
Words          : 132,726


### Defining the World

In [8]:
# Define the WorldState variables -- SET A ( not used anymore ... hence commented out )
#
# Eight variables currently describe the simulated world.
# Additional variables may be added if required.
# All variables begin at index = 100 in W0.
# ------------------------------------------------------------
'''
worldVariables_SetA = [

    WorldVariable(
        name="Oil Price",
        description=(
            "International crude oil price level, represented "
            "by the Brent benchmark."
        ),
        direction="Higher index means higher oil price."
    ),

    WorldVariable(
        name="Hormuz Oil Flow",
        description=(
            "Volume of crude oil and petroleum products successfully "
            "transiting the Strait of Hormuz."
        ),
        direction="Higher index means greater oil flow."
    ),

    WorldVariable(
        name="India Gulf Oil Imports",
        description=(
            "Volume of crude oil imported by India from "
            "Gulf-origin oil-producing countries."
        ),
        direction=(
            "Higher index means greater Indian imports "
            "of Gulf-origin oil."
        )
    ),

    WorldVariable(
        name="Iran Sanctions Enforcement",
        description=(
            "Degree to which restrictions on Iranian petroleum exports, "
            "trade and financial transactions are effectively enforced "
            "rather than circumvented."
        ),
        direction=(
            "Higher index means more effective sanctions enforcement "
            "and less successful circumvention."
        )
    ),

    WorldVariable(
        name="US/Allied Cohesion",
        description=(
            "Degree of alignment between the United States and cooperating "
            "states in their response to Iran and the Gulf crisis."
        ),
        direction="Higher index means greater US/allied cohesion."
    ),

    WorldVariable(
        name="Iran Internal Cohesion",
        description=(
            "Degree of alignment within Iran's political, military and "
            "security establishment in responding to the crisis."
        ),
        direction="Higher index means greater internal Iranian cohesion."
    ),

    WorldVariable(
        name="US-Iran Tension",
        description=(
            "Intensity of strategic hostility between the United States "
            "and Iran across diplomatic, economic and military domains."
        ),
        direction="Higher index means greater US-Iran tension."
    ),

    WorldVariable(
        name="Regional Armed Conflict",
        description=(
            "Intensity and geographic breadth of kinetic military activity "
            "associated with the regional crisis."
        ),
        direction=(
            "Higher index means more intense or geographically "
            "widespread armed conflict."
        )
    )
]
'''

'\nworldVariables_SetA = [\n\n    WorldVariable(\n        name="Oil Price",\n        description=(\n            "International crude oil price level, represented "\n            "by the Brent benchmark."\n        ),\n        direction="Higher index means higher oil price."\n    ),\n\n    WorldVariable(\n        name="Hormuz Oil Flow",\n        description=(\n            "Volume of crude oil and petroleum products successfully "\n            "transiting the Strait of Hormuz."\n        ),\n        direction="Higher index means greater oil flow."\n    ),\n\n    WorldVariable(\n        name="India Gulf Oil Imports",\n        description=(\n            "Volume of crude oil imported by India from "\n            "Gulf-origin oil-producing countries."\n        ),\n        direction=(\n            "Higher index means greater Indian imports "\n            "of Gulf-origin oil."\n        )\n    ),\n\n    WorldVariable(\n        name="Iran Sanctions Enforcement",\n        description=(\n            

In [9]:
# Define the WorldState variables  -- SET B
#
# Eight variables currently describe the simulated world.
# Additional variables may be added if required.
# All variables begin at index = 100 in W0.
# ------------------------------------------------------------
worldVariables_SetB = [

    WorldVariable(
        name="Oil Price",
        description=(
            "International crude oil price level, represented "
            "by the Brent benchmark."
        ),
        direction=(
            "Higher index means higher oil price."
        )
    ),

    WorldVariable(
        name="Hormuz Oil Flow",
        description=(
            "Volume of crude oil and petroleum products successfully "
            "transiting the Strait of Hormuz."
        ),
        direction=(
            "Higher index means greater oil flow."
        )
    ),

    WorldVariable(
        name="Regional Diplomatic Engagement",
        description=(
            "Degree of active diplomatic engagement aimed at managing, "
            "containing or resolving the regional crisis, including "
            "direct negotiations, mediation, backchannels and "
            "de-escalation mechanisms."
        ),
        direction=(
            "Higher index means greater diplomatic engagement "
            "and stronger functioning channels for de-escalation."
        )
    ),

    WorldVariable(
        name="Economic Pressure on Iran",
        description=(
            "Degree of economic pressure on Iran arising from sanctions, "
            "trade and financial restrictions, blockade measures, "
            "export constraints and related economic coercion."
        ),
        direction=(
            "Higher index means greater economic pressure on Iran."
        )
    ),

    WorldVariable(
        name="US/Regional Partner Cohesion",
        description=(
            "Degree of alignment between the United States and key "
            "regional partners, including Israel, Saudi Arabia and "
            "other Gulf states, in responding to Iran and the "
            "regional crisis."
        ),
        direction=(
            "Higher index means greater alignment and willingness "
            "to coordinate with US policy."
        )
    ),

    WorldVariable(
        name="Iran Domestic Stability",
        description=(
            "Degree of stability and resilience within Iran's political, "
            "military, security and domestic system under the pressures "
            "generated by the regional crisis."
        ),
        direction=(
            "Higher index means greater Iranian domestic stability."
        )
    ),

    WorldVariable(
        name="US-Iran Tension",
        description=(
            "Intensity of strategic hostility between the United States "
            "and Iran across diplomatic, economic and military domains."
        ),
        direction=(
            "Higher index means greater US-Iran tension."
        )
    ),

    WorldVariable(
        name="Regional Armed Conflict",
        description=(
            "Intensity and geographic breadth of kinetic military activity "
            "associated with the regional crisis."
        ),
        direction=(
            "Higher index means more intense or geographically "
            "widespread armed conflict."
        )
    )

]

In [10]:
# Set B of WorldVariables is used
#
#
worldVariables = worldVariables_SetB

## Agents and their Roles

### ZeitWorld

In [11]:
# Load ZeitWorld roles and prompts
#
# ZW_Role                  : defines ZeitWorld's role and discipline
# ZW_Initialisation_Prompt : creates W0 from the WorldFactBase
# ZW_Update_Prompt         : evolves the current WorldState after an Event
# ------------------------------------------------------------

ZW_Role = getPrompt(f"{ZW_PATH}/ZW_Role_v0.txt")
ZW_Initialisation_Prompt = getPrompt(f"{ZW_PATH}/ZW_Initialisation_Prompt_v1.txt")
ZW_Update_Prompt = getPrompt(f"{ZW_PATH}/ZW_Update_Prompt_v0.txt")

# Instantiate ZeitWorld
#
# ZeitWorld is created with its LLM client, model and governing role.
# The initialised flag confirms whether W0 has already been created.
# ------------------------------------------------------------

ZeitWorld = ZeitWorld_Agent(client, model, ZW_Role)

### Chanakya

In [12]:
#!ls -l "{CH_PATH}"

In [13]:
# Load Chanakya roles and strategy
#
# Generic Role : defines the common strategic discipline of all Chanakyas
# US Role      : defines US objectives, interests and constraints
# Iran Role    : defines Iranian objectives, interests and constraints
# BinaryHD     : generates two contrasting Hawk and Dove candidates
# ============================================================

CH_Generic_Role = getPrompt(f"{CH_PATH}/CH_Role_Generic_v0.txt")
CH_US_Role      = getPrompt(f"{CH_PATH}/CH_Role_US_v0.txt")
CH_IR_Role      = getPrompt(f"{CH_PATH}/CH_Role_Iran_v0.txt")

#CH_Strategy     = getPrompt(f"{CH_PATH}/CH_Strategy_BinaryHD_v0.txt")
#CH_Strategy     = getPrompt(f"{CH_PATH}/CH_Strategy_Flex2_v261009.txt")
CH_Strategy     = getPrompt(f"{CH_PATH}/CH_Strategy_Flex2_v261009_A.txt")

# Instantiate Chanakya agents
#
# Each Chanakya represents a national actor with its own
# interests and constraints, using the current Binary Hawk/Dove strategy.
# The registry maps each actor to its corresponding Chanakya.
# ============================================================

Chanakya_US = Chanakya_Agent(
    agent_id="ChanakyaUS_HD",
    actor="US",
    generic_role=CH_Generic_Role,
    specific_role=CH_US_Role,
    strategy=CH_Strategy
)

Chanakya_US.show()

Chanakya_IR = Chanakya_Agent(
    agent_id="ChanakyaIR_HD",
    actor="Iran",
    generic_role=CH_Generic_Role,
    specific_role=CH_IR_Role,
    strategy=CH_Strategy
)

Chanakya_IR.show()

ChanakyaRegistry = {
    "US": Chanakya_US,
    "Iran": Chanakya_IR
}

Chanakya : ChanakyaUS_HD
Actor    : US
Chanakya : ChanakyaIR_HD
Actor    : Iran


### Centaur

In [14]:
# Load Centaur roles
#
# Generic Role : defines the common adjudication discipline of all Centaurs
# UNSG Role    : provides a fair, independent UN diplomatic perspective
# ============================================================

CT_Generic_Role = getPrompt(f"{CT_PATH}/CT_Generic_Role_v0.txt")
CT_UNSG_Role      = getPrompt(f"{CT_PATH}/CT_UNSG_Role_v0.txt")

# Instantiate Centaur adjudicator
#
# CentaurUNSG evaluates Chanakya's Candidate Events from the
# perspective of a fair and independent UN Secretary-General.
# ============================================================
Centaur_UNSG = Centaur_Agent(
    agent_id="CentaurUNSG",
    generic_role=CT_Generic_Role,
    adjudication_role=CT_UNSG_Role
)

### Reporter

In [15]:
# ============================================================
# Load PressReporter Roles and Prompts
#
# Generic Role defines reporting discipline.
# Narrative Prompt produces the human-readable Scenario history.
# Structured Prompt produces the machine-oriented Markdown record.
# ============================================================

PR_Generic_Role = getPrompt(f"{PR_PATH}/PR_Generic_Role_v0.txt")

#PR_Narrative    = getPrompt(f"{PR_PATH}/PR_Narrative_v0.txt")
#PR_Narrative    = getPrompt(f"{PR_PATH}/PR_Narrative_Brief_v2.txt")
PR_Narrative    = getPrompt(f"{PR_PATH}/PR_Narrative_Brief_v3.txt")

PR_Structured   = getPrompt(f"{PR_PATH}/PR_Structured_v0.txt")


# ============================================================
# Instantiate Human Reporter
# ============================================================

Reporter = Reporter_Agent(
    client=client,
    model=model,
    agent_id="Vyasa",
    generic_role=PR_Generic_Role
)

## Initialisation of Scenario

In [16]:
# Initialise Simulation State
#
# W stores the evolving WorldStates of the simulation.
# E stores the sequence of Events defining the Scenario.
# W0_Template defines the empty structure from which W0 is created.
# ------------------------------------------------------------

E = []
W = []

W0_Template = WorldState(
    state_id="W0",
    variables=worldVariables
)

# Initialise the first WorldState
#
# ZeitWorld uses the WorldFactBase and W0 template to create W0.
# The completed WorldState is automatically appended to W as W[0].
# ============================================================

_ = ZeitWorld.initialise(W,W0_Template, WorldFactBase, ZW_Initialisation_Prompt)

#print(ZeitWorld.initialised)

Initialising WorldState ... 
Model: gpt-5.6-luna | Effort: medium | Input: 182,289 | Output: 1,837 | Reasoning: 266 | Total: 184,126


In [17]:
# A first look at the intial World State W[0]
# The initial values are all set to 100, as the baseline
#

#W[0].show(-2)  # ---- just the world vectors
#W[0].show()    # ---- the entire state
#W[0].show(4)    # ---- details of any one vector

In [18]:
# Define initial Trigger options
#
# Two alternative Events are available to start the simulation.
# One is manually selected as E0; thereafter Events are generated
# and selected by the simulation.
# ============================================================

Trigger1 = Event(
    actor="Iran",
    action=(
        "Iran declares the Strait of Hormuz closed to all commercial shipping, "
        "orders its forces to prevent tanker transit, and launches attacks against "
        "US naval forces operating in the Gulf."
    ),
    rationale = "Initial Trigger for Iran"
)


Trigger2 = Event(
    actor="US",
    action=(
        "The United States announces an immediate suspension of offensive military "
        "operations against Iran and offers direct negotiations on maritime security "
        "and sanctions relief, while maintaining defensive protection of US forces "
        "and commercial shipping in the Gulf."
    ),
    rationale = "Initial Trigger for US"
)

# ============================================================================

Triggers = {
    "TR1": Trigger1,
    "TR2": Trigger2
}

In [19]:
# Select the initial Trigger
#
# The manually selected DefaultTrigger starts the first epoch.
# Its selection weight is None because it was not chosen by Shakuni.
# ============================================================
#DefaultTrigger = Trigger1
#DefaultTrigger = Trigger2
#DefaultTrigger.show()
#print(DefaultTrigger.selection_weight)


In [20]:
# Initialise / Reset Scenario
#
# Retains W0 and clears all subsequent WorldStates and Events,
# restoring the simulation to its starting condition.
# ============================================================

def resetWorld():
    del W[1:]
    E.clear()

    print("Scenario reset to W0")

# ------------------------------------------------------------

resetWorld()

Scenario reset to W0


### Define Single Epoch

In [21]:
#-------------------------------------------------
#
# An Event and the resultant WorldState is an Epoch
#
#-------------------------------------------------
def runEpoch():
    # ============================================================
    # EPOCH STEP 1 — Establish Current Trigger
    #
    # The epoch is derived from the current WorldState history.
    # For the first epoch, DefaultTrigger becomes E0.
    # Thereafter, the last Reaction becomes the next Trigger.
    # ============================================================

    epoch = len(W) - 1
    if len(E) == 0:
        DefaultTrigger.event_id = f"E{epoch}"
        E.append(DefaultTrigger)
    else:
        assert E[epoch] is E[-1]

    #E[epoch].show()

    #print("epoch :", epoch, "len(W):", len(W), "len(E):", len(E), " ***** Step # 1 complete" )
    #print("Epoch ",epoch, " Starts ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")

    # ============================================================
    # EPOCH STEP 2 — Update WorldState
    #
    # ZeitWorld applies Trigger E[epoch] to W[epoch] and creates
    # the next WorldState W[epoch+1].
    # ============================================================

    _ = ZeitWorld.update( W,E[epoch],ZW_Update_Prompt)

    #E[epoch].show()
    #W[epoch+1].show(-2)

    #print("epoch :", epoch, "len(W):", len(W), "len(E):", len(E), " ***** Step # 2 complete" )

    # ============================================================
    # EPOCH STEP 3 — Select Active Actor
    #
    # The Turn Controller identifies which actor responds to the
    # Current Trigger and activates its corresponding Chanakya.
    # ============================================================

    #ActiveChanakya = getActiveChanakya(E[-1], ChanakyaRegistry)
    ActiveChanakya = getActiveChanakya(E[epoch], ChanakyaRegistry)
    #ActiveChanakya.show()

    #print("epoch :", epoch, "len(W):", len(W), "len(E):", len(E), " ***** Step # 3 complete" )

    # ============================================================
    # EPOCH STEP 4 — Generate Candidate Reactions
    #
    # The Active Chanakya evaluates the updated WorldState and
    # generates Candidate Events as possible reactions to the Trigger.
    # ============================================================

    Candidates = ActiveChanakya.propose(
        client=client,
        model=model,
        trigger=E[epoch],
        world=W[epoch+1]
    )

    #for i, candidate in enumerate(Candidates):
    #    print(f"\nCandidate Reaction {i+1}")
    #    print("-" * 60)
    #    candidate.show()

    #print("epoch :", epoch, "len(W):", len(W), "len(E):", len(E), " ***** Step # 4 complete" )

    # ============================================================
    # EPOCH STEP 5 — Adjudicate Candidate Reactions
    #
    # Centaur evaluates the Candidate Events against the current
    # WorldState and assigns each a relative simulation weight.
    # ============================================================

    weights = Centaur_UNSG.evaluate(
        client,
        model,
        E[epoch],
        W[epoch + 1],
        Candidates
    )

    #print("\nCentaur Weights")
    #print("----------------")
    #print("Weights",weights)

    #print("epoch :", epoch, "len(W):", len(W), "len(E):", len(E), " ***** Step # 5 complete" )

    # ============================================================
    #
    # Centaur weights represent the relative conditional likelihood
    # of each Candidate at this decision point.
    #
    # Shakuni makes a single stochastic selection using these weights.
    # The selected Candidate becomes the Reaction, retains its Centaur
    # weight as selection_weight, receives the next Event ID, and is
    # appended to E as the Trigger for the next epoch.
    # ============================================================

    Reaction = Shakuni(Candidates, weights)

    Reaction.event_id = f"E{len(E)}"
    E.append(Reaction)

    #print("\nShakuni has spoken")
    #print("------------------")
    #Reaction.show()
    #print(Reaction.selection_weight)

    print("epoch :", epoch, "len(W):", len(W), "len(E):", len(E) )
    print("Epoch ends .............................................")



##Scenario | Multiple Epochs

In [22]:
#-------------------------------------------------
#
# Collection of Epochs is a Scenario
#
#-------------------------------------------------

def runScenario(MaxEvents = 3,_scenarioID = "dummy", RunSuffix = "000000-000000"):

    #print("Scenario ",_scenarioID, " starts ----------------------------")

    resetWorld()
    while len(E) < MaxEvents:
        runEpoch()

    # Apply the final Event and close the Scenario
    _ = ZeitWorld.update(W, E[-1], ZW_Update_Prompt)

    # Structural integrity check
    assert len(E) == MaxEvents
    assert len(W) == MaxEvents + 1
    assert len(W) == len(E) + 1

    print("Scenario ", _scenarioID," complete with ", "Events :", len(E), "WorldStates :", len(W))

    #print("Now Generating Report for scenario ",_scenarioID )


    # ============================================================
    # Generate Narrative Scenario Report
    #
    # PressReporter converts the completed Scenario (W + E) into a
    # human-readable narrative using the Narrative reporting prompt.
    # The resulting report is then formatted and saved as a Word document.
    # ============================================================
    '''
    NarrativeReport = Reporter.GenerateReport( W,E, PR_Narrative)
    ReportTitle="Middle East Oil Security — Scenario Brief"
    NarrativeFile = createNarrativeDocx( NarrativeReport, W, E, _scenarioID,ReportTitle,ReportSuffix )
    print(f"Narrative Scenario Report written to: {NarrativeFile}")
    '''
    # ============================================================
    # Generate Structured Scenario Report
    #
    # Produces a structured Markdown representation of the completed
    # Scenario for subsequent machine-based analysis.
    #
    # ============================================================

    #StructuredReport = Reporter.GenerateReport( W, E, PR_Structured)
    StructuredReport = GenerateStructuredReport(W, E)
    StructuredFile = f"MOS_{_scenarioID}_{ReportSuffix}.md"

    with open(StructuredFile, "w", encoding="utf-8") as f:
        f.write(StructuredReport)

    print(f"Structured Scenario Report written to: {StructuredFile}")
    print("Scenario ends ------------------------------------------------------------")
    #display(Markdown(StructuredReport))



##Ensemble | Multiple Scenarios

In [23]:
#-------------------------------------------------
#
# Collection of Scenarios is an Ensemble
#
#-------------------------------------------------

# Select the initial Trigger from the registry
# ============================================================
TriggerID = "TR1"
DefaultTrigger = Triggers[TriggerID]
#DefaultTrigger.show()

#
MaxEvents = 3
NumOfRuns = 5
StartRun = 1
EndRun = StartRun + NumOfRuns - 1


ScenarioID = f"{TriggerID}ME{MaxEvents}"

for run in range(StartRun, EndRun + 1):
    print("Run # ",run, '===================================================')
    now = datetime.now(ZoneInfo("Asia/Kolkata"))
    ReportSuffix = now.strftime('%y%m%d_%H%M%S')
    runScenario(
        MaxEvents=MaxEvents,
        _scenarioID=f"{ScenarioID}_R{run:02d}",
        RunSuffix = ReportSuffix
    )

Run #  1 ===================================================
Scenario reset to W0
Updating WorldState ---------- W0
Model: gpt-5.6-luna | Effort: medium | Input: 2,886 | Output: 2,111 | Reasoning: 191 | Total: 4,997
Appended  WorldState ---------- W1
Chanakya working ---------- ChanakyaUS_HD
Model: gpt-5.6-luna | Effort: medium | Input: 2,188 | Output: 266 | Reasoning: 57 | Total: 2,454
Centaur working ---------- CentaurUNSG
Model: gpt-5.6-luna | Effort: medium | Input: 1,942 | Output: 59 | Reasoning: 43 | Total: 2,001
epoch : 0 len(W): 2 len(E): 2
Epoch ends .............................................
Updating WorldState ---------- W1
Model: gpt-5.6-luna | Effort: medium | Input: 3,240 | Output: 2,228 | Reasoning: 106 | Total: 5,468
Appended  WorldState ---------- W2
Chanakya working ---------- ChanakyaIR_HD
Model: gpt-5.6-luna | Effort: medium | Input: 2,367 | Output: 226 | Reasoning: 0 | Total: 2,593
Centaur working ---------- CentaurUNSG
Model: gpt-5.6-luna | Effort: medium | Inp

In [24]:
# 1. Scenario identifier
print(ScenarioID)

# 2. Display destination
print("Drive Reports Directory:", OP_PATH)

# 3. Show existing files and folders
!ls -lh "{OP_PATH}"

# 4. Create timestamped directory using IST
NEW_DIR = f"{OP_PATH}/{ScenarioID}_$(TZ=Asia/Kolkata date +%y%m%d_%H%M%S)"
!mkdir -p "{NEW_DIR}"

# 5. Copy MOS reports from /content
!cp -v /content/MOS_TR*.* "{NEW_DIR}/"

# 6. Verify copied files
!ls -lh "{NEW_DIR}"

TR1ME3
Drive Reports Directory: /content/drive/MyDrive/Praxis/Course Geopolitics/Centaur Project/MOS//Reports
total 12K
drwx------ 2 root root 4.0K Oct  9 06:45 TR1ME3_261009_121557
drwx------ 2 root root 4.0K Oct  9 06:03 TR2ME4_261009_113341
drwx------ 2 root root 4.0K Oct  9 06:16 TR2ME4_261009_114633
'/content/MOS_TR1ME3_R01_261009_123939.md' -> '/content/drive/MyDrive/Praxis/Course Geopolitics/Centaur Project/MOS//Reports/TR1ME3_261009_124607/MOS_TR1ME3_R01_261009_123939.md'
'/content/MOS_TR1ME3_R02_261009_124056.md' -> '/content/drive/MyDrive/Praxis/Course Geopolitics/Centaur Project/MOS//Reports/TR1ME3_261009_124607/MOS_TR1ME3_R02_261009_124056.md'
'/content/MOS_TR1ME3_R03_261009_124215.md' -> '/content/drive/MyDrive/Praxis/Course Geopolitics/Centaur Project/MOS//Reports/TR1ME3_261009_124607/MOS_TR1ME3_R03_261009_124215.md'
'/content/MOS_TR1ME3_R04_261009_124334.md' -> '/content/drive/MyDrive/Praxis/Course Geopolitics/Centaur Project/MOS//Reports/TR1ME3_261009_124607/MOS_TR1ME3_

### Debugger

In [25]:
#print(E)

#ChronoTantra | ChronoYantra | ChronoMantra | ChronoSutra
 [More Information ](http://bit.ly/chronobooks) <br>

[![More Information](https://raw.githubusercontent.com/prithwis/parashar21/refs/heads/main/images/CTYMS-Colab-Footer.png)](https://chronos.yantrajaal.com)